[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BASE-Laboratory/OpenImpala/blob/main/notebooks/profiling_and_tuning.ipynb)

# OpenImpala Interactive Profiling & Tuning

This notebook profiles solver performance, tunes AMReX grid decomposition, and compares scaling behavior — all within Google Colab's free tier (T4 GPU, ~12 GB RAM).

**Sections:**
1. Synthetic Dataset Generation
2. Baseline Validation
3. Solver Profiling Sweep
4. Grid Size Sweep (`max_grid_size`)
5. Dataset Size Scaling
6. Direction Anisotropy Check
7. Multi-Porosity Comparison
8. **AMReX TinyProfiler Breakdown** *(solver setup vs solve vs flux)*
9. **NVIDIA Nsight Systems GPU Profiling** *(kernel-level timeline)*
10. Summary & HPC Recommendations

## Setup

Detect hardware, install dependencies, and initialize the OpenImpala session.

In [ ]:
# Detect GPU
import subprocess
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=10)
    gpu_name = result.stdout.strip()
    if gpu_name:
        print(f"GPU detected: {gpu_name}")
    else:
        print("No GPU detected — running CPU-only.")
except FileNotFoundError:
    gpu_name = ""
    print("nvidia-smi not found — running CPU-only.")

In [ ]:
# Install system MPI and Python packages
!apt-get install -y libopenmpi-dev > /dev/null 2>&1
!pip install openimpala-cuda --find-links https://github.com/BASE-Laboratory/OpenImpala/releases/latest/download/[all] > /dev/null 2>&1
!pip install porespy > /dev/null 2>&1
print("Dependencies installed.")

In [ ]:
import openimpala as oi
from openimpala import core as oic
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print(f"OpenImpala version: {oi.__version__}")

In [ ]:
# Initialize OpenImpala session (kept open across cells)
session = oi.Session()
session.__enter__()
print("OpenImpala session started.")

## Section 1: Synthetic Dataset Generation

We use [PoreSpy](https://porespy.org/) to generate 3D porous microstructures at multiple sizes for profiling sweeps.

In [ ]:
import porespy as ps


def generate_microstructure(size, porosity=0.5, seed=42):
    """Generate a synthetic 3D porous medium using porespy."""
    np.random.seed(seed)
    im = ps.generators.blobs(shape=[size] * 3, porosity=porosity, blobiness=1.5)
    return im.astype(np.int32)  # 0 = pore, 1 = solid

In [ ]:
data_small = generate_microstructure(64)
data_medium = generate_microstructure(128)
data_large = generate_microstructure(256)

for name, data in [("small (64³)", data_small),
                    ("medium (128³)", data_medium),
                    ("large (256³)", data_large)]:
    vf = np.sum(data == 0) / data.size
    print(f"{name:16s}  shape={str(data.shape):20s}  porosity={vf:.3f}")

In [ ]:
# Visualize a single Z-slice of data_medium
fig, ax = plt.subplots(figsize=(6, 6))
mid_z = data_medium.shape[0] // 2
ax.imshow(data_medium[mid_z, :, :], cmap="gray", origin="lower")
ax.set_title(f"data_medium — Z-slice {mid_z} (white=solid, black=pore)")
ax.set_xlabel("X")
ax.set_ylabel("Y")
plt.tight_layout()
plt.show()

## Section 2: Baseline — Quick Validation

Run a quick sanity check on `data_small` to confirm the solver pipeline works.

In [ ]:
%%time
vf = oi.volume_fraction(data_small, phase=0)
perc = oi.percolation_check(data_small, phase=0, direction="z")
result = oi.tortuosity(data_small, phase=0, direction="z", solver="pcg")

print(f"Volume fraction (pore): {vf.fraction:.4f}")
print(f"Percolating (Z):        {perc.percolates}")
print(f"Tortuosity (Z):         {result.tortuosity:.4f}")
print(f"Solver converged:       {result.solver_converged}")
print(f"Iterations:             {result.iterations}")

## Section 3: Solver Profiling Sweep

Profile all available HYPRE solvers on `data_medium` (128³). Each solver is run 3 times; we report the median wall time.

In [ ]:
solvers = ["pcg", "flexgmres", "gmres", "bicgstab", "smg", "pfmg", "mlmg"]
n_repeats = 3
solver_records = []

for solver_name in solvers:
    times = []
    last_result = None
    failed = False

    for trial in range(n_repeats):
        try:
            t0 = time.perf_counter()
            res = oi.tortuosity(data_medium, phase=0, direction="z",
                                solver=solver_name, max_grid_size=32, verbose=0)
            t1 = time.perf_counter()
            times.append(t1 - t0)
            last_result = res
        except Exception as e:
            print(f"  {solver_name} trial {trial}: FAILED — {e}")
            failed = True
            break

    if failed or last_result is None:
        solver_records.append({
            "solver": solver_name, "wall_time_s": np.nan,
            "iterations": np.nan, "residual_norm": np.nan,
            "tortuosity": np.nan, "converged": False
        })
    else:
        solver_records.append({
            "solver": solver_name,
            "wall_time_s": np.median(times),
            "iterations": last_result.iterations,
            "residual_norm": last_result.residual_norm,
            "tortuosity": last_result.tortuosity,
            "converged": last_result.solver_converged
        })
    status = "OK" if not failed else "FAILED"
    print(f"  {solver_name:12s} — {status}")

df_solvers = pd.DataFrame(solver_records)
df_solvers

In [ ]:
# Dual-axis bar chart: wall time (bars) + iteration count (dots)
fig, ax1 = plt.subplots(figsize=(10, 5))

colors = ["#2ecc71" if c else "#e74c3c" for c in df_solvers["converged"]]
y_pos = np.arange(len(df_solvers))
bars = ax1.barh(y_pos, df_solvers["wall_time_s"], color=colors, alpha=0.85,
                edgecolor="white", linewidth=0.8)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(df_solvers["solver"])
ax1.set_xlabel("Wall Time (s)", color="#2c3e50")
ax1.set_title("Solver Profiling — 128³ dataset", fontsize=13, fontweight="bold")
ax1.tick_params(axis="x", colors="#2c3e50")

# Second x-axis for iteration count
ax2 = ax1.twiny()
ax2.plot(df_solvers["iterations"], y_pos, "D", color="#8e44ad", markersize=8,
         zorder=5, label="Iterations")
ax2.set_xlabel("Iterations", color="#8e44ad")
ax2.tick_params(axis="x", colors="#8e44ad")

# Convergence legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#2ecc71", label="Converged"),
                   Patch(facecolor="#e74c3c", label="Failed"),
                   plt.Line2D([0], [0], marker="D", color="#8e44ad", linestyle="None",
                              markersize=8, label="Iterations")]
ax1.legend(handles=legend_elements, loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
# Select the best solver (fastest converged)
df_converged = df_solvers[df_solvers["converged"]]
best_solver = df_converged.loc[df_converged["wall_time_s"].idxmin(), "solver"]
print(f"Best solver: {best_solver}")

## Section 4: Grid Size Sweep (`max_grid_size`)

The `max_grid_size` parameter controls AMReX box decomposition. Smaller boxes improve CPU cache locality and MPI parallelism; larger boxes give better GPU kernel saturation with less overhead.

In [ ]:
grid_sizes = [8, 16, 32, 64, 128]
grid_records = []

for gs in grid_sizes:
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        res = oi.tortuosity(data_medium, phase=0, direction="z",
                            solver=best_solver, max_grid_size=gs, verbose=0)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    grid_records.append({
        "max_grid_size": gs,
        "wall_time_s": np.median(times),
        "tortuosity": res.tortuosity
    })
    print(f"  max_grid_size={gs:4d}  time={np.median(times):.3f}s  τ={res.tortuosity:.4f}")

df_grid = pd.DataFrame(grid_records)
df_grid

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_grid["max_grid_size"], df_grid["wall_time_s"], "o-", linewidth=2, markersize=8)
ax.set_xlabel("max_grid_size")
ax.set_ylabel("Wall Time (s)")
ax.set_title(f"Grid Decomposition Sweep — 128³, solver={best_solver}")

# Annotations
ax.annotate("Smaller boxes →\nbetter cache locality,\nmore MPI parallelism",
            xy=(grid_sizes[0], df_grid["wall_time_s"].iloc[0]),
            xytext=(grid_sizes[1], df_grid["wall_time_s"].max() * 0.9),
            arrowprops=dict(arrowstyle="->", color="gray"), fontsize=9, color="gray")
ax.annotate("Larger boxes →\nbetter GPU saturation,\nless overhead",
            xy=(grid_sizes[-1], df_grid["wall_time_s"].iloc[-1]),
            xytext=(grid_sizes[-2], df_grid["wall_time_s"].max() * 0.9),
            arrowprops=dict(arrowstyle="->", color="gray"), fontsize=9, color="gray")

plt.tight_layout()
plt.show()

In [ ]:
best_grid_size = int(df_grid.loc[df_grid["wall_time_s"].idxmin(), "max_grid_size"])
print(f"Optimal max_grid_size: {best_grid_size}")

### Solver x Grid Size Heatmap

Sweep all converged solvers across all grid sizes to reveal the full 2D parameter space.

In [ ]:
# Solver x Grid Size heatmap — full 2D parameter sweep
converged_solvers = df_solvers[df_solvers["converged"]]["solver"].tolist()
heatmap_grid_sizes = [8, 16, 32, 64, 128]
heatmap_data = np.full((len(converged_solvers), len(heatmap_grid_sizes)), np.nan)

for i, s in enumerate(converged_solvers):
    for j, gs in enumerate(heatmap_grid_sizes):
        try:
            t0 = time.perf_counter()
            oi.tortuosity(data_medium, phase=0, direction="z",
                          solver=s, max_grid_size=gs, verbose=0)
            heatmap_data[i, j] = time.perf_counter() - t0
        except Exception:
            heatmap_data[i, j] = np.nan
    print(f"  {s} done")

fig, ax = plt.subplots(figsize=(9, max(4, len(converged_solvers) * 0.7)))
im = ax.imshow(heatmap_data, cmap="YlOrRd", aspect="auto")

# Annotate each cell with its value
for i in range(heatmap_data.shape[0]):
    for j in range(heatmap_data.shape[1]):
        val = heatmap_data[i, j]
        if not np.isnan(val):
            text_color = "white" if val > np.nanmedian(heatmap_data) else "black"
            ax.text(j, i, f"{val:.2f}s", ha="center", va="center",
                    fontsize=9, fontweight="bold", color=text_color)

ax.set_xticks(range(len(heatmap_grid_sizes)))
ax.set_xticklabels(heatmap_grid_sizes)
ax.set_yticks(range(len(converged_solvers)))
ax.set_yticklabels(converged_solvers)
ax.set_xlabel("max_grid_size")
ax.set_ylabel("Solver")
ax.set_title("Solver x Grid Size — Wall Time (s)", fontsize=13, fontweight="bold")

# Mark the global minimum
best_idx = np.unravel_index(np.nanargmin(heatmap_data), heatmap_data.shape)
ax.add_patch(plt.Rectangle((best_idx[1] - 0.5, best_idx[0] - 0.5), 1, 1,
             fill=False, edgecolor="#2ecc71", linewidth=3, linestyle="--"))
ax.text(best_idx[1], best_idx[0] + 0.35, "BEST", ha="center", va="top",
        fontsize=7, color="#2ecc71", fontweight="bold")

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Wall Time (s)")
plt.tight_layout()
plt.show()

## Section 5: Dataset Size Scaling

Measure how wall time scales with problem size using the best solver and grid configuration. Ideal scaling is ~O(N) for multigrid solvers and ~O(N log N) for Krylov methods.

In [ ]:
datasets = [("64³", data_small, 64), ("128³", data_medium, 128), ("256³", data_large, 256)]
size_records = []

for label, data, size in datasets:
    times = []
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        res = oi.tortuosity(data, phase=0, direction="z",
                            solver=best_solver, max_grid_size=best_grid_size, verbose=0)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    median_t = np.median(times)
    size_records.append({
        "label": label, "size": size, "n_voxels": size**3,
        "wall_time_s": median_t, "tortuosity": res.tortuosity
    })
    print(f"  {label}  time={median_t:.3f}s  τ={res.tortuosity:.4f}")

df_size = pd.DataFrame(size_records)
df_size

In [ ]:
# Log-log plot with filled area under the scaling curve
log_n = np.log10(df_size["n_voxels"].values.astype(float))
log_t = np.log10(df_size["wall_time_s"].values)
coeffs = np.polyfit(log_n, log_t, 1)
exponent = coeffs[0]

# Dense interpolation for smooth fill
n_dense = np.logspace(log_n.min(), log_n.max(), 200)
t_dense = 10 ** np.polyval(coeffs, np.log10(n_dense))

fig, ax = plt.subplots(figsize=(9, 5))

# Filled area under measured curve (gradient effect via stacked fills)
from matplotlib.colors import LinearSegmentedColormap
n_vals = df_size["n_voxels"].values.astype(float)
t_vals = df_size["wall_time_s"].values
ax.fill_between(n_dense, t_dense, alpha=0.15, color="#3498db")
ax.fill_between(n_vals, t_vals, alpha=0.25, color="#3498db", step="mid")

# Power-law fit line
ax.loglog(n_dense, t_dense, "--", color="#95a5a6", linewidth=1.5,
          label=f"Power-law fit: O(N$^{{{exponent:.2f}}}$)")

# Measured data points (prominent)
ax.loglog(n_vals, t_vals, "o-", color="#2c3e50", linewidth=2.5,
          markersize=12, markerfacecolor="#3498db", markeredgecolor="#2c3e50",
          markeredgewidth=2, label="Measured", zorder=5)

ax.set_xlabel("Number of Voxels", fontsize=11)
ax.set_ylabel("Wall Time (s)", fontsize=11)
ax.set_title(f"Size Scaling — solver={best_solver}, max_grid_size={best_grid_size}",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=10, framealpha=0.9)

for _, row in df_size.iterrows():
    ax.annotate(row["label"], (row["n_voxels"], row["wall_time_s"]),
                textcoords="offset points", xytext=(12, 8), fontsize=10,
                fontweight="bold", color="#2c3e50",
                arrowprops=dict(arrowstyle="->", color="#bdc3c7", lw=1.2))

plt.tight_layout()
plt.show()
print(f"Scaling exponent: {exponent:.2f}")

## Section 6: Direction Anisotropy Check

For isotropic synthetic blobs, tortuosity should be roughly equal in all three directions. Significant anisotropy would indicate preferred transport paths in the microstructure.

In [ ]:
directions = ["x", "y", "z"]
tau_values = {}

for d in directions:
    res = oi.tortuosity(data_medium, phase=0, direction=d,
                        solver=best_solver, max_grid_size=best_grid_size, verbose=0)
    tau_values[d] = res.tortuosity
    print(f"  τ_{d} = {res.tortuosity:.4f}")

In [ ]:
# Radar (spider) chart for directional anisotropy
labels = [f"$\\tau_{d}$" for d in directions]
values = [tau_values[d] for d in directions]

# Close the polygon
angles = np.linspace(0, 2 * np.pi, len(directions), endpoint=False).tolist()
values_closed = values + [values[0]]
angles_closed = angles + [angles[0]]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

# Filled radar area
ax.fill(angles_closed, values_closed, color="#3498db", alpha=0.2)
ax.plot(angles_closed, values_closed, "o-", color="#3498db", linewidth=2.5,
        markersize=10, markerfacecolor="white", markeredgecolor="#3498db",
        markeredgewidth=2.5)

# Value labels at each vertex
for angle, val, d in zip(angles, values, directions):
    ax.annotate(f"{val:.3f}", xy=(angle, val),
                textcoords="offset points", xytext=(8, 8),
                fontsize=11, fontweight="bold", color="#2c3e50")

# Isotropic reference circle (mean tortuosity)
tau_mean = np.mean(values)
ref_angles = np.linspace(0, 2 * np.pi, 100)
ax.plot(ref_angles, [tau_mean] * len(ref_angles), "--", color="#e74c3c",
        linewidth=1.5, alpha=0.6, label=f"Isotropic ref (τ={tau_mean:.3f})")

ax.set_xticks(angles)
ax.set_xticklabels([f"$\\tau_{{{d.upper()}}}$" for d in directions], fontsize=13)
ax.set_title("Direction Anisotropy — 128³ isotropic blobs\n",
             fontsize=13, fontweight="bold")
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1), framealpha=0.9)

plt.tight_layout()
plt.show()

## Section 7: Multi-Porosity Comparison

Generate datasets at varying porosities and compare measured tortuosity against the Bruggeman relation (τ = ε^−0.5), a common approximation for isotropic porous media.

In [ ]:
porosities = [0.3, 0.4, 0.5, 0.6, 0.7]
porosity_records = []

for p in porosities:
    data_p = generate_microstructure(128, porosity=p)
    vf = oi.volume_fraction(data_p, phase=0)

    try:
        perc = oi.percolation_check(data_p, phase=0, direction="z", verbose=0)
        if not perc.percolates:
            print(f"  ε={p:.1f} — not percolating, skipped")
            porosity_records.append({
                "porosity": p, "volume_fraction": vf.fraction,
                "tortuosity": np.nan, "percolates": False
            })
            continue
    except Exception:
        print(f"  ε={p:.1f} — percolation check failed, skipped")
        porosity_records.append({
            "porosity": p, "volume_fraction": vf.fraction,
            "tortuosity": np.nan, "percolates": False
        })
        continue

    res = oi.tortuosity(data_p, phase=0, direction="z",
                        solver=best_solver, max_grid_size=best_grid_size, verbose=0)
    porosity_records.append({
        "porosity": p, "volume_fraction": vf.fraction,
        "tortuosity": res.tortuosity, "percolates": True
    })
    print(f"  ε={p:.1f}  VF={vf.fraction:.3f}  τ={res.tortuosity:.4f}")

df_porosity = pd.DataFrame(porosity_records)
df_porosity

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

# Bruggeman reference
eps_ref = np.linspace(0.2, 0.8, 100)
tau_brugg = eps_ref ** (-0.5)
ax.plot(eps_ref, tau_brugg, "--", color="gray", linewidth=2, label="Bruggeman (τ = ε⁻⁰·⁵)")

# Measured values
df_valid = df_porosity[df_porosity["percolates"]]
ax.plot(df_valid["porosity"], df_valid["tortuosity"], "o-", linewidth=2,
        markersize=10, color="#e74c3c", label="Measured (OpenImpala)")

# Mark non-percolating
df_noperc = df_porosity[~df_porosity["percolates"]]
if not df_noperc.empty:
    ax.scatter(df_noperc["porosity"], [0.5] * len(df_noperc), marker="x",
               s=100, color="black", zorder=5, label="Non-percolating")

ax.set_xlabel("Porosity (ε)")
ax.set_ylabel("Tortuosity (τ)")
ax.set_title("Tortuosity vs. Porosity — 128³ synthetic microstructures")
ax.legend()
plt.tight_layout()
plt.show()

The Bruggeman relation (τ = ε⁻⁰·⁵) is a simple effective-medium approximation. Real (or realistic synthetic) microstructures typically deviate due to pore connectivity, bottlenecks, and geometric tortuosity effects that the Bruggeman model does not capture.

## Section 8: AMReX TinyProfiler — Internal Breakdown

AMReX ships with a built-in **TinyProfiler** that hooks into every `BL_PROFILE`-annotated function in the C++ source. OpenImpala instruments 24 key functions — solver setup, matrix assembly, HYPRE solve, flux computation, and more.

When the AMReX session finalizes, TinyProfiler prints a breakdown table to stdout. We capture this output by restarting the session with verbose output, running a solve, then parsing the profiler dump.

**This section answers #31's core questions:**
- Solver iterations vs. setup time
- Matrix assembly vs. linear solve
- Flux computation vs. everything else

In [ ]:
# Close the current session so TinyProfiler dumps its output,
# then restart with verbose output to capture the profiler table.

import io
import sys
import re

# Close existing session cleanly
session.__exit__(None, None, None)

def run_with_profiler(data, solver="pcg", direction="z", max_grid_size=32):
    """Run a solve in a fresh AMReX session and capture TinyProfiler output."""
    # Capture stdout during the entire session lifecycle
    captured = io.StringIO()
    old_stdout = sys.stdout

    # Start fresh session
    s = oi.Session()
    s.__enter__()

    # Run the solve with verbose=1 to get internal timing prints
    res = oi.tortuosity(data, phase=0, direction=direction,
                        solver=solver, max_grid_size=max_grid_size, verbose=1)

    # Finalize — this is when TinyProfiler prints its summary
    sys.stdout = captured
    s.__exit__(None, None, None)
    sys.stdout = old_stdout

    return res, captured.getvalue()

# Run profiled solve on 128³ dataset
print(f"Running profiled solve (128³, solver={best_solver})...")
prof_result, prof_output = run_with_profiler(data_medium, solver=best_solver)
print(f"Tortuosity: {prof_result.tortuosity:.4f}")
print(f"Converged:  {prof_result.solver_converged}")
print(f"\nTinyProfiler output length: {len(prof_output)} chars")

# Show raw output for inspection
if prof_output.strip():
    print("\n--- Raw TinyProfiler Output ---")
    print(prof_output[:3000])
else:
    print("\nNote: TinyProfiler output may be empty if AMReX was built without")
    print("TINY_PROFILE support. The wall-time profiling below still works.")

In [ ]:
def parse_tiny_profiler(output):
    """Parse AMReX TinyProfiler output into a DataFrame.

    TinyProfiler prints tables like:
        Name                          NCalls  Excl. Min  Excl. Avg  Excl. Max   Max %
        TortuosityHypre::solve             1     0.45       0.45       0.45     52.3%
    """
    records = []
    # Match lines with function name, ncalls, and timing columns
    pattern = re.compile(
        r"^\s*(\S+.*\S)\s+"       # function name
        r"(\d+)\s+"               # NCalls
        r"([\d.e+-]+)\s+"         # Excl. Min
        r"([\d.e+-]+)\s+"         # Excl. Avg
        r"([\d.e+-]+)\s+"         # Excl. Max
        r"([\d.e+-]+)\s*%",       # Max %
        re.MULTILINE
    )
    for m in pattern.finditer(output):
        records.append({
            "function": m.group(1).strip(),
            "ncalls": int(m.group(2)),
            "excl_min_s": float(m.group(3)),
            "excl_avg_s": float(m.group(4)),
            "excl_max_s": float(m.group(5)),
            "max_pct": float(m.group(6)),
        })
    return pd.DataFrame(records)

df_prof = parse_tiny_profiler(prof_output)

if len(df_prof) > 0:
    df_prof = df_prof.sort_values("excl_avg_s", ascending=False).head(15)
    print(f"Top {len(df_prof)} functions by exclusive time:")
    print(df_prof.to_string(index=False))
else:
    # Fall back to wall-time breakdown if TinyProfiler isn't available
    print("TinyProfiler data not available — using wall-time phase breakdown instead.")
    print("Running phase-by-phase timing...")

    # Restart session for manual timing
    session = oi.Session()
    session.__enter__()

    # Phase 1: Volume fraction + percolation check (pre-solve)
    t0 = time.perf_counter()
    vf = oi.volume_fraction(data_medium, phase=0)
    perc = oi.percolation_check(data_medium, phase=0, direction="z")
    t_presolve = time.perf_counter() - t0

    # Phase 2: Full tortuosity solve (includes setup + matrix fill + solve + flux)
    t0 = time.perf_counter()
    res = oi.tortuosity(data_medium, phase=0, direction="z",
                        solver=best_solver, max_grid_size=best_grid_size, verbose=0)
    t_solve_total = time.perf_counter() - t0

    # Phase 3: Repeat at different sizes to estimate setup vs compute scaling
    t0 = time.perf_counter()
    res_small = oi.tortuosity(data_small, phase=0, direction="z",
                              solver=best_solver, max_grid_size=best_grid_size, verbose=0)
    t_solve_small = time.perf_counter() - t0

    # Estimate: setup overhead ≈ time that doesn't scale with problem size
    # If 64³ takes t1 and 128³ takes t2, then t2/t1 ≈ scaling factor
    # The "flat" part (setup + HYPRE init) can be estimated
    df_prof = pd.DataFrame([
        {"phase": "Pre-solve checks", "time_s": t_presolve},
        {"phase": f"Full solve (128³)", "time_s": t_solve_total},
        {"phase": f"Full solve (64³)", "time_s": t_solve_small},
    ])
    scale_ratio = t_solve_total / max(t_solve_small, 1e-9)
    print(f"\n  Pre-solve (VF + percolation): {t_presolve:.4f}s")
    print(f"  Full solve 64³:               {t_solve_small:.4f}s")
    print(f"  Full solve 128³:              {t_solve_total:.4f}s")
    print(f"  Scaling ratio (128/64):       {scale_ratio:.2f}x")

In [ ]:
# Visualize the profiler breakdown
if "max_pct" in df_prof.columns and len(df_prof) > 0:
    # TinyProfiler horizontal bar chart
    fig, ax = plt.subplots(figsize=(10, max(4, len(df_prof) * 0.4)))

    # Color by category
    def categorize(name):
        name_l = name.lower()
        if "solve" in name_l:
            return "#e74c3c"  # red
        elif any(k in name_l for k in ["setup", "fill", "matrix", "stencil"]):
            return "#f39c12"  # orange
        elif any(k in name_l for k in ["flux", "plane"]):
            return "#3498db"  # blue
        elif any(k in name_l for k in ["mask", "precondition", "flood", "percolation"]):
            return "#2ecc71"  # green
        else:
            return "#95a5a6"  # gray

    colors = [categorize(f) for f in df_prof["function"]]
    y_pos = np.arange(len(df_prof))

    ax.barh(y_pos, df_prof["excl_avg_s"], color=colors, alpha=0.85,
            edgecolor="white", linewidth=0.8)
    for i, (_, row) in enumerate(df_prof.iterrows()):
        ax.text(row["excl_avg_s"] + 0.001, i, f'{row["max_pct"]:.1f}%',
                va="center", fontsize=9, color="#2c3e50")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_prof["function"], fontsize=9)
    ax.set_xlabel("Exclusive Time (s)")
    ax.set_title("AMReX TinyProfiler — Function Breakdown", fontsize=13, fontweight="bold")
    ax.invert_yaxis()

    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor="#e74c3c", label="Linear solve"),
        Patch(facecolor="#f39c12", label="Matrix assembly/setup"),
        Patch(facecolor="#3498db", label="Flux computation"),
        Patch(facecolor="#2ecc71", label="Pre-processing"),
        Patch(facecolor="#95a5a6", label="Other"),
    ]
    ax.legend(handles=legend_elements, loc="lower right", framealpha=0.9)
    plt.tight_layout()
    plt.show()

    # Pie chart of time categories
    categories = {"Linear solve": 0, "Matrix assembly": 0,
                  "Flux computation": 0, "Pre-processing": 0, "Other": 0}
    for _, row in df_prof.iterrows():
        name_l = row["function"].lower()
        if "solve" in name_l:
            categories["Linear solve"] += row["excl_avg_s"]
        elif any(k in name_l for k in ["setup", "fill", "matrix", "stencil"]):
            categories["Matrix assembly"] += row["excl_avg_s"]
        elif any(k in name_l for k in ["flux", "plane"]):
            categories["Flux computation"] += row["excl_avg_s"]
        elif any(k in name_l for k in ["mask", "precondition", "flood", "percolation"]):
            categories["Pre-processing"] += row["excl_avg_s"]
        else:
            categories["Other"] += row["excl_avg_s"]

    cat_colors = ["#e74c3c", "#f39c12", "#3498db", "#2ecc71", "#95a5a6"]
    fig, ax = plt.subplots(figsize=(7, 7))
    wedges, texts, autotexts = ax.pie(
        categories.values(), labels=categories.keys(), colors=cat_colors,
        autopct="%1.1f%%", startangle=90, textprops={"fontsize": 11})
    for t in autotexts:
        t.set_fontweight("bold")
    ax.set_title("Time Distribution by Category", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

else:
    # Fallback: bar chart from manual phase timing
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    phases = df_prof["phase"].tolist()
    times_s = df_prof["time_s"].tolist()
    colors = ["#2ecc71", "#e74c3c", "#3498db"]

    ax1.barh(phases, times_s, color=colors[:len(phases)], alpha=0.85)
    ax1.set_xlabel("Wall Time (s)")
    ax1.set_title("Phase Timing Breakdown", fontweight="bold")

    if len(times_s) >= 3:
        ax2.bar(["64\u00b3", "128\u00b3"], [times_s[2], times_s[1]], color="#e74c3c", alpha=0.85)
        ax2.bar(["64\u00b3", "128\u00b3"], [times_s[0], times_s[0]],
                bottom=[times_s[2], times_s[1]], color="#2ecc71", alpha=0.85,
                label="Pre-solve overhead")
        ax2.set_ylabel("Wall Time (s)")
        ax2.set_title("Setup vs Compute Scaling", fontweight="bold")
        ax2.legend()

    plt.tight_layout()
    plt.show()

## Section 9: NVIDIA Nsight Systems — GPU Kernel Profiling

If a GPU is available, we use **NVIDIA Nsight Systems** (`nsys`) to capture a
detailed timeline of CUDA kernel launches, memory transfers, and API calls.
This reveals:

- **GPU kernel occupancy** — are kernels large enough to saturate the GPU?
- **Host-device transfers** — how much time is spent copying data?
- **Kernel launch overhead** — are there too many tiny kernel launches?
- **Idle gaps** — is the GPU waiting on CPU work between kernels?

> Colab provides `nsys` pre-installed on GPU runtimes. On HPC clusters,
> load it via `module load nsight-systems`.

In [ ]:
import shutil
import os

nsys_available = shutil.which("nsys") is not None and bool(gpu_name)

if nsys_available:
    print(f"nsys found: {shutil.which('nsys')}")
    print(f"GPU: {gpu_name}")
    print("\nRunning Nsight Systems profile...")

    # Write a self-contained profiling script
    script = '''
import openimpala as oi
import numpy as np
import porespy as ps

np.random.seed(42)
data = ps.generators.blobs(shape=[128, 128, 128], porosity=0.5, blobiness=1.5)
data = data.astype(np.int32)

with oi.Session():
    res = oi.tortuosity(data, phase=0, direction="z", solver="pcg",
                        max_grid_size=64, verbose=0)
    print(f"tau={res.tortuosity:.4f} converged={res.solver_converged}")
'''
    with open('/tmp/oi_profile.py', 'w') as f:
        f.write(script)

    # Run nsys profile — capture CUDA API, kernels, and memory ops
    !nsys profile \
        --output=/tmp/oi_profile \
        --force-overwrite=true \
        --trace=cuda,nvtx,osrt \
        --cuda-memory-usage=true \
        --stats=true \
        python3 /tmp/oi_profile.py 2>&1 | tail -80

    print("\nProfile saved to /tmp/oi_profile.nsys-rep")
    print("Download and open in Nsight Systems GUI for interactive analysis.")
else:
    if not gpu_name:
        print("No GPU detected — skipping Nsight Systems profiling.")
        print("To enable: Runtime > Change runtime type > T4 GPU")
    else:
        print("nsys not found — install via: apt-get install nsight-systems")
    print("\nSkipping to summary.")

In [ ]:
if nsys_available:
    # Parse the nsys stats output for CUDA kernel summary
    nsys_stats = subprocess.run(
        ["nsys", "stats", "--report", "cuda_gpu_kern_sum",
         "--format", "csv", "/tmp/oi_profile.nsys-rep"],
        capture_output=True, text=True, timeout=60
    )

    if nsys_stats.returncode == 0 and nsys_stats.stdout.strip():
        # Parse CSV output
        from io import StringIO
        import csv

        lines = nsys_stats.stdout.strip().split('\n')
        # Find the header line (contains 'Time')
        header_idx = None
        for i, line in enumerate(lines):
            if 'Time' in line and 'Name' in line:
                header_idx = i
                break

        if header_idx is not None:
            csv_text = '\n'.join(lines[header_idx:])
            df_kernels = pd.read_csv(StringIO(csv_text))
            # Show top kernels by total time
            if 'Total Time (ns)' in df_kernels.columns:
                df_kernels['Total Time (ms)'] = df_kernels['Total Time (ns)'] / 1e6
                df_kernels = df_kernels.sort_values('Total Time (ns)', ascending=False).head(10)
                print("Top 10 CUDA Kernels by Total GPU Time:")
                print(df_kernels[['Name', 'Total Time (ms)', 'Instances']].to_string(index=False))
            else:
                print("Kernel summary:")
                print(df_kernels.head(10).to_string(index=False))
        else:
            print("Could not parse nsys kernel stats.")
            print(nsys_stats.stdout[:2000])
    else:
        print("nsys stats returned no kernel data.")
        if nsys_stats.stderr:
            print(nsys_stats.stderr[:500])

In [ ]:
if nsys_available:
    # Also get memory transfer stats
    nsys_mem = subprocess.run(
        ["nsys", "stats", "--report", "cuda_gpu_mem_size_sum",
         "--format", "csv", "/tmp/oi_profile.nsys-rep"],
        capture_output=True, text=True, timeout=60
    )

    if nsys_mem.returncode == 0 and nsys_mem.stdout.strip():
        lines = nsys_mem.stdout.strip().split('\n')
        header_idx = None
        for i, line in enumerate(lines):
            if 'Operation' in line or 'Total' in line:
                header_idx = i
                break
        if header_idx is not None:
            csv_text = '\n'.join(lines[header_idx:])
            df_mem = pd.read_csv(StringIO(csv_text))
            print("\nCUDA Memory Transfer Summary:")
            print(df_mem.to_string(index=False))
        else:
            print("\nMemory transfer data:")
            print(nsys_mem.stdout[:1000])

    # Visualize kernel distribution if we have data
    try:
        if 'df_kernels' in dir() and len(df_kernels) > 0 and 'Total Time (ms)' in df_kernels.columns:
            fig, ax = plt.subplots(figsize=(10, max(3, len(df_kernels) * 0.35)))

            # Shorten kernel names for display
            short_names = []
            for n in df_kernels['Name']:
                # Extract the last meaningful part of mangled kernel names
                parts = str(n).split('::')
                short = parts[-1] if len(parts) > 1 else str(n)
                short = short[:50] + '...' if len(short) > 50 else short
                short_names.append(short)

            y_pos = np.arange(len(df_kernels))
            ax.barh(y_pos, df_kernels['Total Time (ms)'].values,
                    color='#e67e22', alpha=0.85, edgecolor='white')
            ax.set_yticks(y_pos)
            ax.set_yticklabels(short_names, fontsize=8)
            ax.set_xlabel('Total GPU Time (ms)')
            ax.set_title('CUDA Kernel Time Distribution', fontsize=13, fontweight='bold')
            ax.invert_yaxis()
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print(f"Could not plot kernel distribution: {e}")

    print("\nFor interactive analysis, download /tmp/oi_profile.nsys-rep")
    print("and open it in the Nsight Systems GUI (free from NVIDIA).")

### Interpreting Nsight Systems results

**Key things to look for:**

| Pattern | Meaning | Action |
|---------|---------|--------|
| Many tiny kernels (<1 ms each) | Kernel launch overhead dominates | Increase `max_grid_size` to fuse work |
| Large H2D/D2H transfers | Data copying bottleneck | Check if arrays are being copied repeatedly |
| Long gaps between kernels | CPU is the bottleneck | Look for serial CPU work between GPU calls |
| One dominant kernel (>80% time) | Solver kernel is the hot path | Focus optimization there (expected for large problems) |
| Even distribution across kernels | Setup/assembly is significant | Consider caching matrix assembly across solves |

**On HPC clusters** with multiple GPUs, combine `nsys` with MPI profiling:
```bash
mpirun -np 4 nsys profile --output=profile_rank%q{OMPI_COMM_WORLD_RANK} \
    openimpala inputs_256.txt
```

## Section 10: Summary & HPC Recommendations

In [ ]:
# Estimated time for 256³ from measured data
time_256 = df_size.loc[df_size["size"] == 256, "wall_time_s"].values[0]

print("=" * 60)
print("  RECOMMENDED CONFIGURATION")
print("=" * 60)
print(f"  Best solver:          {best_solver}")
print(f"  Optimal max_grid_size: {best_grid_size}")
print(f"  Time per 256³ solve:  {time_256:.2f} s")

# Memory regime estimate
for label, size in [("256³", 256), ("400³", 400), ("512³", 512)]:
    mem_gb = (size ** 3 * 8 * 3) / 1e9  # rough: 3 arrays of float64
    regime = "safe" if mem_gb < 4 else ("caution" if mem_gb < 10 else "DANGER")
    print(f"  Memory {label}: ~{mem_gb:.1f} GB — [{regime}]")
print("=" * 60)

In [ ]:
# HPC scaling estimates (extrapolate from power-law fit)
print("\nHPC Scaling Estimates (extrapolated from T4 measurements):")
print("-" * 50)
print(f"{'Size':>8s}  {'Voxels':>12s}  {'Est. Time (s)':>14s}")
print("-" * 50)

for size in [256, 512, 1024, 2048]:
    n = size ** 3
    est_time = 10 ** np.polyval(coeffs, np.log10(n))
    print(f"{size:>5d}³  {n:>12,d}  {est_time:>13.1f}")

print("-" * 50)
print("Note: These are rough estimates from a single T4.")
print("Real HPC clusters with A100 GPUs will be significantly faster.")

In [ ]:
# Template inputs file for HPC
inputs_template = f"""# Recommended OpenImpala inputs (generated by profiling notebook)
filename = your_image.tif
phase_id = 0
direction = ALL
solver_type = {best_solver}
box_size = {best_grid_size}
calculation_method = flow_through
hypre.maxiter = 1000
hypre.eps = 1e-9
physics.type = diffusion
verbose = 1
"""

print("Recommended HPC inputs file:")
print("=" * 50)
print(inputs_template)

## Cleanup

Close the OpenImpala session.

In [ ]:
# Clean up — session may already be closed from TinyProfiler section
try:
    from openimpala._core import amrex_initialized
    if amrex_initialized():
        session.__exit__(None, None, None)
        print("OpenImpala session closed.")
    else:
        print("Session already finalized (closed during TinyProfiler section).")
except Exception:
    print("Session cleanup complete.")